# Hyperparameter Optimization

Optimize model hyperparameters with Optuna.

In [ ]:
import sys
sys.path.insert(0, '..')

import optuna
from pytorch_lightning import seed_everything
from src.data import GestureDataModule
from src.models import BiLSTMModule, TransformerModule
from src.optimization import OptunaObjective, BiLSTMSearchSpace, TransformerSearchSpace, create_study

## Configuration

In [ ]:
# Choose model
MODEL = 'bilstm'  # 'bilstm' or 'transformer'

# Optimization settings
N_TRIALS = 50
MAX_EPOCHS = 30
PATIENCE = 10

# Cross-validation
USE_CV = True
N_FOLDS = 5

# Data
DATA_PATH = '../data/DYLEM-GRID'
SEED = 42

seed_everything(SEED)

## Setup

In [ ]:
dm = GestureDataModule(data_path=DATA_PATH, seed=SEED)

MODEL_CLASS = BiLSTMModule if MODEL == 'bilstm' else TransformerModule
SEARCH_SPACE = BiLSTMSearchSpace() if MODEL == 'bilstm' else TransformerSearchSpace()

print(f'Model: {MODEL.upper()}')
print(f'Trials: {N_TRIALS}')
print(f'CV: {N_FOLDS}-fold' if USE_CV else 'Single split')

## Run Optimization

In [ ]:
study = create_study(f'{MODEL}_optimization')

objective = OptunaObjective(
    model_class=MODEL_CLASS,
    datamodule=dm,
    search_space=SEARCH_SPACE,
    use_cv=USE_CV,
    n_folds=N_FOLDS,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

## Results

In [ ]:
print('=' * 50)
print('OPTIMIZATION RESULTS')
print('=' * 50)
print(f'Best trial: #{study.best_trial.number}')
print(f'Best accuracy: {study.best_trial.value:.4f}')
print('\nBest hyperparameters:')
for k, v in study.best_trial.params.items():
    print(f'  {k}: {v}')

## Visualizations

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

plot_optimization_history(study)

In [ ]:
if len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]) >= 10:
    plot_param_importances(study)

In [ ]:
plot_parallel_coordinate(study)

## Save Results

In [ ]:
import json, joblib
from pathlib import Path

save_dir = Path(f'../results/optuna/{MODEL}')
save_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(study, save_dir / 'study.pkl')

with open(save_dir / 'results.json', 'w') as f:
    json.dump({
        'best_accuracy': study.best_trial.value,
        'best_params': study.best_trial.params,
        'n_trials': len(study.trials)
    }, f, indent=2)

print(f'Results saved to {save_dir}')